In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import re
from collections import Counter
from sklearn.model_selection import train_test_split
import time
import random

In [2]:
# 1. Load & Preprocess
df = pd.read_csv('all_songs_data_50k.csv')

In [3]:
# --- Konfigurasi Model & Pelatihan ---
CONFIG = {
    # Dimensi Embedding untuk Teks
    "text_embedding_dim": 150, # dinaikkan
    "hidden_dim": 300, # dinaikkan
    "n_layers": 2,
    "dropout": 0.5, # disesuaikan
    # Dimensi Output Embedding (HARUS SAMA untuk kedua menara)
    "output_embedding_dim": 64,
    "batch_size": 128,
    "learning_rate": 0.001,
    "epochs": 25, # dinaikkan untuk konvergensi yang lebih baik
    "max_len": 50,
    "margin": 0.2,
    "min_song_freq": 5, # Batas frekuensi minimum untuk lagu
}

In [4]:
# --- 1. Persiapan Device (CPU/GPU) ---
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"GPU terdeteksi: {torch.cuda.get_device_name(0)}. Pelatihan akan menggunakan GPU.")
else:
    device = torch.device("cpu")
    print("GPU tidak ditemukan. Pelatihan akan menggunakan CPU.")

GPU terdeteksi: NVIDIA GeForce GTX 1650 Ti. Pelatihan akan menggunakan GPU.


In [5]:
def clean_text(text):
    stopwords = set([
        'yang', 'dan', 'di', 'ke', 'dari', 'untuk', 'dengan', 'pada', 'adalah', 'itu', 'ini', 'saya', 'aku', 'kamu', 'dia', 'kita', 'mereka', 'sebagai', 'juga', 'karena', 'atau', 'tapi', 'tidak', 'ya', 'oh', 'lagi', 'sudah', 'belum', 'akan', 'bisa', 'harus', 'dapat', 'saja', 'jadi', 'oleh', 'apa', 'siapa', 'mengapa', 'bagaimana', 'semua', 'setiap', 'hanya', 'saat', 'dalam', 'ada', 'pun', 'lah', 'punya', 'buat', 'sama', 'kalau', 'sampai', 'seperti', 'tentang', 'tanpa', 'namun', 'lebih', 'kurang', 'masih', 'baru', 'lama', 'begitu', 'begini', 'mau', 'pernah', 'selalu', 'sering', 'jarang', 'bahwa', 'oleh', 'atas', 'bawah', 'antara', 'sebelum', 'sesudah', 'hingga', 'sejak', 'selama', 'demi', 'guna', 'terhadap', 'kepada', 'dalam', 'lu', 'gua', 'lo', 'gw', 'gue', 'kok', 'nih', 'deh', 'dong', 'lah', 'pun', 'mah', 'si', 'ya', 'kan', 'nggak', 'ngga', 'enggak', 'ga', 'gak', 'ngga', 'nggak', 'nggk', 'aja', 'doang', 'tok', 'toh', 'lah', 'pun', 'mah', 'si', 'ya', 'kan', 'nggak', 'ngga', 'enggak', 'ga', 'gak', 'ngga', 'nggak', 'nggk', 'aja', 'doang', 'tok', 'toh'
    ])
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    text = ' '.join([w for w in text.split() if w not in stopwords])
    return text

def build_vocab(all_texts):
    word_counts = Counter(word for text in all_texts for word in text.split())
    vocab = {word: i+2 for i, (word, _) in enumerate(word_counts.most_common())}
    vocab['<PAD>'] = 0
    vocab['<UNK>'] = 1
    return vocab

In [6]:
# --- 3. Architecture Model ---

# Tower Pertama(message)
class MessageTower(nn.Module):
    def __init__(self, vocab_size, config):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, config["text_embedding_dim"], padding_idx=0)
        self.gru = nn.GRU(
            config["text_embedding_dim"],
            config["hidden_dim"],
            num_layers=config["n_layers"],
            dropout=config["dropout"],
            batch_first=True
        )
        # Layer terakhir untuk memproyeksikan output GRU ke dimensi embedding akhir
        self.fc = nn.Linear(config["hidden_dim"], config["output_embedding_dim"])

    def forward(self, x):
        embedded = self.embedding(x)
        _, hidden = self.gru(embedded)
        hidden = hidden[-1] # Ambil hidden state terakhir dari layer terakhir
        # Normalisasi L2 pada output embedding adalah praktik yang baik
        return F.normalize(self.fc(hidden), p=2, dim=1)

# Tower Kedua(song)
class SongTower(nn.Module):
    def __init__(self, num_songs, config):
        super().__init__()
        # Cukup sebuah layer embedding untuk merepresentasikan setiap lagu secara unik
        self.embedding = nn.Embedding(num_songs, config["output_embedding_dim"])

    def forward(self, x):
        # Normalisasi L2 pada output embedding
        return F.normalize(self.embedding(x), p=2, dim=1)

In [7]:
# --- 4. Dataset dan DataLoader ---
class RecommendationDataset(Dataset):
    def __init__(self, messages, song_data, vocab, song_map, max_len):
        self.messages = messages
        self.song_data = song_data # Berisi nama lagu (string)
        self.vocab = vocab
        self.song_map = song_map
        self.max_len = max_len

    def __len__(self):
        return len(self.messages)

    def __getitem__(self, idx):
        # Ambil pasangan (pesan, lagu) yang positif
        message = self.messages[idx]
        song_info = self.song_data[idx]  # FIX: gunakan string, jangan tuple
        
        # Vectorize pesan
        vectorized_msg = [self.vocab.get(word, self.vocab['<UNK>']) for word in message.split()]
        if len(vectorized_msg) < self.max_len:
            vectorized_msg += [self.vocab['<PAD>']] * (self.max_len - len(vectorized_msg))
        else:
            vectorized_msg = vectorized_msg[:self.max_len]
        
        # Dapatkan ID lagu dari map
        song_id = self.song_map[song_info]
        
        return torch.tensor(vectorized_msg, dtype=torch.long), torch.tensor(song_id, dtype=torch.long)

In [8]:
# --- 5. Fungsi Pelatihan ---
def train_recommendation_model(message_tower, song_tower, train_loader, config, device):
    message_tower.to(device)
    song_tower.to(device)
    
    # Gabungkan parameter dari kedua model untuk optimizer
    optimizer = optim.AdamW(
        list(message_tower.parameters()) + list(song_tower.parameters()),
        lr=config["learning_rate"]
    )
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)
    # Loss function yang dirancang untuk tugas ini
    criterion = nn.CosineEmbeddingLoss(margin=config["margin"])

    all_song_ids = torch.arange(len(song_map)).to(device)
    for epoch in range(1, config["epochs"] + 1):
        start_time = time.time()
        message_tower.train()
        song_tower.train()
        total_loss = 0
        
        for messages, positive_songs in train_loader:
            messages, positive_songs = messages.to(device), positive_songs.to(device)
            
            optimizer.zero_grad()
            
            # Dapatkan embeddings dari kedua menara
            message_embeddings = message_tower(messages)
            song_embeddings = song_tower(positive_songs)
            
            # --- Strategi In-Batch Negative Sampling ---
            # Kita gunakan lagu dari item lain di batch yang sama sebagai sampel negatif.
            # Ini sangat efisien.
            
            # Loss untuk pasangan positif (target = 1)
            loss_pos = criterion(message_embeddings, song_embeddings, torch.ones(messages.size(0)).to(device))
            
            # Loss untuk pasangan negatif (target = -1)
            # Kita pasangkan message i dengan song j, di mana i != j
            # Cara termudah adalah dengan menggeser tensor lagu
            negative_song_embeddings = torch.roll(song_embeddings, shifts=1, dims=0)
            loss_neg = criterion(message_embeddings, negative_song_embeddings, -torch.ones(messages.size(0)).to(device))
            
            # Random hard negative sampling
            rand_neg_ids = torch.randint(0, len(song_map), (messages.size(0),)).to(device)
            rand_neg_emb = song_tower(rand_neg_ids)
            loss_rand_neg = criterion(message_embeddings, rand_neg_emb, -torch.ones(messages.size(0)).to(device))
            
            # Loss total adalah jumlah dari keduanya
            loss = loss_pos + loss_neg + 0.5 * loss_rand_neg
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()

        scheduler.step()
        avg_loss = total_loss / len(train_loader)
        end_time = time.time()
        print(f"Epoch: {epoch}/{config['epochs']} | "
              f"Waktu: {end_time - start_time:.2f}s | "
              f"Average Loss: {avg_loss:.4f} | LR: {scheduler.get_last_lr()[0]:.6f}")

In [9]:
# --- 6. Fungsi untuk Rekomendasi (Inference) ---
def build_song_index(song_tower, song_map, device):
    """Proses semua lagu untuk membuat matriks embedding (indeks)."""
    print("\nMembangun indeks embedding untuk semua lagu...")
    song_tower.eval()
    with torch.no_grad():
        # Buat tensor berisi semua ID lagu
        all_song_ids = torch.arange(len(song_map)).to(device)
        # Dapatkan embedding untuk semua lagu sekaligus
        song_index_embeddings = song_tower(all_song_ids)
    print("Indeks berhasil dibangun.")
    return song_index_embeddings

def recommend_songs(message_text, message_tower, song_index, vocab, config, device, top_k=10):
    """Memberikan top_k rekomendasi id lagu untuk sebuah pesan."""
    message_tower.eval()
    # 1. Proses pesan input menjadi embedding
    cleaned_text = clean_text(message_text)
    vectorized = [vocab.get(word, vocab['<UNK>']) for word in cleaned_text.split()]
    if len(vectorized) < config["max_len"]:
        vectorized += [vocab['<PAD>']] * (config["max_len"] - len(vectorized))
    else:
        vectorized = vectorized[:config["max_len"]]
    query_tensor = torch.tensor(vectorized, dtype=torch.long).unsqueeze(0).to(device)
    with torch.no_grad():
        query_embedding = message_tower(query_tensor)
    # 2. Hitung kemiripan (cosine similarity) antara query dan semua lagu di indeks
    similarities = F.cosine_similarity(query_embedding, song_index)
    # 3. Dapatkan top_k lagu dengan similarity tertinggi
    top_results = torch.topk(similarities, k=top_k)
    # 4. Kembalikan id lagu dan skornya
    recommendations = []
    for score, song_idx in zip(top_results.values, top_results.indices):
        recommendations.append({
            "song_id": song_idx.item(),
            "similarity_score": score.item()
        })
    return recommendations

# MAIN EXECUTION


In [10]:
# --- Pra-pemrosesan Data Asli ---
# Hapus baris dengan nilai kosong di kolom penting
df.dropna(subset=['message', 'song_name', 'song_artist'], inplace=True)
# Hapus pesan atau nama lagu yang hanya berisi spasi
df = df[df['message'].str.strip() != '']
df = df[df['song_name'].str.strip() != '']
df.reset_index(drop=True, inplace=True)

# Filter lagu yang jarang muncul untuk mengurangi noise
song_counts = df['song_name'].value_counts()
frequent_songs = song_counts[song_counts >= CONFIG["min_song_freq"]].index
df = df[df['song_name'].isin(frequent_songs)].reset_index(drop=True)

print(f"Dataset dimuat dan dibersihkan. Jumlah data setelah filtering: {len(df)}")

df['cleaned_message'] = df['message'].apply(clean_text)

Dataset dimuat dan dibersihkan. Jumlah data setelah filtering: 45041


In [11]:
# Split data menjadi train dan validasi
from sklearn.model_selection import train_test_split
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['song_name'])
print(f"Train: {len(train_df)}, Validasi: {len(val_df)}")


Train: 36032, Validasi: 9009


In [12]:
# 2. Build Vocab dan Song Mapping (hanya dari data train)
vocab = build_vocab(train_df['cleaned_message'])
all_songs = train_df['song_name'].unique().tolist()
song_map = {song: i for i, song in enumerate(all_songs)}

vocab_size = len(vocab)
num_songs = len(all_songs)
print(f"\nUkuran Vocabulary: {vocab_size}, Jumlah Lagu Unik: {num_songs}")


Ukuran Vocabulary: 72909, Jumlah Lagu Unik: 1334


In [13]:
# 3. Create Dataset & DataLoader
X_train = train_df['cleaned_message'].values
y_train = train_df['song_name'].values
X_val = val_df['cleaned_message'].values
y_val = val_df['song_name'].values

train_dataset = RecommendationDataset(X_train, y_train, vocab, song_map, CONFIG["max_len"])
val_dataset = RecommendationDataset(X_val, y_val, vocab, song_map, CONFIG["max_len"])
train_loader = DataLoader(train_dataset, batch_size=CONFIG["batch_size"], shuffle=True)

In [14]:
# 4. Inisialisasi Model
message_tower = MessageTower(vocab_size, CONFIG)
song_tower = SongTower(num_songs, CONFIG)

In [15]:
# 5. Latih Model
train_recommendation_model(message_tower, song_tower, train_loader, CONFIG, device)

Epoch: 1/25 | Waktu: 14.49s | Average Loss: 0.8985 | LR: 0.001000
Epoch: 2/25 | Waktu: 14.12s | Average Loss: 0.8462 | LR: 0.001000
Epoch: 2/25 | Waktu: 14.12s | Average Loss: 0.8462 | LR: 0.001000
Epoch: 3/25 | Waktu: 14.14s | Average Loss: 0.8116 | LR: 0.001000
Epoch: 3/25 | Waktu: 14.14s | Average Loss: 0.8116 | LR: 0.001000
Epoch: 4/25 | Waktu: 14.50s | Average Loss: 0.7895 | LR: 0.001000
Epoch: 4/25 | Waktu: 14.50s | Average Loss: 0.7895 | LR: 0.001000
Epoch: 5/25 | Waktu: 14.86s | Average Loss: 0.7733 | LR: 0.000500
Epoch: 5/25 | Waktu: 14.86s | Average Loss: 0.7733 | LR: 0.000500
Epoch: 6/25 | Waktu: 14.64s | Average Loss: 0.7603 | LR: 0.000500
Epoch: 6/25 | Waktu: 14.64s | Average Loss: 0.7603 | LR: 0.000500
Epoch: 7/25 | Waktu: 14.43s | Average Loss: 0.7530 | LR: 0.000500
Epoch: 7/25 | Waktu: 14.43s | Average Loss: 0.7530 | LR: 0.000500
Epoch: 8/25 | Waktu: 14.42s | Average Loss: 0.7478 | LR: 0.000500
Epoch: 8/25 | Waktu: 14.42s | Average Loss: 0.7478 | LR: 0.000500
Epoch: 9/2

In [16]:
# 6. Bangun Indeks untuk Inference
song_index = build_song_index(song_tower, song_map, device)


Membangun indeks embedding untuk semua lagu...
Indeks berhasil dibangun.


In [17]:
# 7. Tes Rekomendasi
print("\n--- Tes Rekomendasi ---")
test_message_1 = "kamu percaya ga sama aku? kamu tau ga sih bagi aku, kamu itu apa"
test_message_2 = "lu gila banget!"

# Buat mapping id ke (nama lagu, artis)
song_id_to_info = df[['song_name', 'song_artist']].drop_duplicates().reset_index(drop=True)

recs_1 = recommend_songs(test_message_1, message_tower, song_index, vocab, CONFIG, device)
print(f"\nRekomendasi untuk pesan: '{test_message_1}'")
for rec in recs_1:
    song_id = rec['song_id']
    if song_id < len(song_id_to_info):
        song_name = song_id_to_info.loc[song_id, 'song_name']
        song_artist = song_id_to_info.loc[song_id, 'song_artist']
        print(f"  - {song_name} | {song_artist} (ID: {song_id}, Score: {rec['similarity_score']:.4f})")
    else:
        print(f"  - Song ID: {song_id} (Score: {rec['similarity_score']:.4f})")

recs_2 = recommend_songs(test_message_2, message_tower, song_index, vocab, CONFIG, device)
print(f"\nRekomendasi untuk pesan: '{test_message_2}'")
for rec in recs_2:
    song_id = rec['song_id']
    if song_id < len(song_id_to_info):
        song_name = song_id_to_info.loc[song_id, 'song_name']
        song_artist = song_id_to_info.loc[song_id, 'song_artist']
        print(f"  - {song_name} | {song_artist} (ID: {song_id}, Score: {rec['similarity_score']:.4f})")
    else:
        print(f"  - Song ID: {song_id} (Score: {rec['similarity_score']:.4f})")


--- Tes Rekomendasi ---

Rekomendasi untuk pesan: 'kamu percaya ga sama aku? kamu tau ga sih bagi aku, kamu itu apa'
  - This Is How It Feels (with Laufey) | d4vd, Laufey (ID: 1, Score: 0.9530)
  - Work Song | Hozier (ID: 111, Score: 0.5127)
  - Paths | NIKI (ID: 95, Score: 0.5034)
  - The One That Got Away | Katy Perry (ID: 3, Score: 0.4688)
  - Always Forever | Cults (ID: 578, Score: 0.4521)
  - right where you left me - bonus track | Taylor Swift (ID: 32, Score: 0.4396)
  - Kau Rumahku | raissa anggiani (ID: 156, Score: 0.4256)
  - You'll Be In My Heart - Spotify Singles | NIKI (ID: 2, Score: 0.4103)
  - Peter | Taylor Swift (ID: 321, Score: 0.3969)
  - Can't Take My Eyes off You | Frankie Valli (ID: 736, Score: 0.3942)

Rekomendasi untuk pesan: 'lu gila banget!'
  - august | Taylor Swift (ID: 6, Score: 0.9962)
  - Multo | Cup of Joe (ID: 4, Score: 0.9854)
  - Outro | M83 (ID: 52, Score: 0.9827)
  - right where you left me - bonus track | Taylor Swift (ID: 32, Score: 0.5088)
  - Yo

In [18]:
# Ekspor model setelah training
# torch.save({
#     'message_tower_state_dict': message_tower.state_dict(),
#     'song_tower_state_dict': song_tower.state_dict(),
#     'vocab': vocab,
#     'song_map': song_map
# }, 'two_tower_songrec_model.pth')
# print("Model berhasil diekspor ke two_tower_songrec_model.pth")


# Data Split dan Pipeline Training/Validasi

Notebook ini telah diubah agar seluruh proses training dan evaluasi mengikuti best practice:

- **Vocab, song_map, dataset, dan DataLoader** hanya dibuat dari data training.
- **Evaluasi** hanya dilakukan pada data validasi.
- **Metrik evaluasi** yang digunakan: Recall@10, Precision@10, dan Hit Rate@10.


# Evaluasi Model pada Data Validasi

Evaluasi dilakukan pada data validasi menggunakan beberapa metrik retrieval:

- **Recall@10**: Persentase pesan di mana lagu ground truth muncul di 10 besar hasil rekomendasi.
- **Precision@10**: Rata-rata proporsi lagu relevan di 10 besar hasil rekomendasi.
- **Hit Rate@10**: Persentase pesan yang minimal 1 lagu relevan muncul di 10 besar.


In [19]:
# 8. Evaluasi Model pada Data Validasi
print("\n--- Evaluasi Model pada Data Validasi ---")
recall_at_10 = []
precision_at_10 = []
hit_at_10 = []
num_samples = min(1000, len(val_df))
sample_idx = np.random.choice(len(val_df), size=num_samples, replace=False)
for idx in sample_idx:
    message = val_df.iloc[idx]['cleaned_message']
    true_song = val_df.iloc[idx]['song_name']
    if true_song not in song_map:
        continue  # skip lagu yang tidak ada di train
    true_song_id = song_map[true_song]
    recs = recommend_songs(message, message_tower, song_index, vocab, CONFIG, device, top_k=10)
    pred_ids = [rec['song_id'] for rec in recs]
    recall = int(true_song_id in pred_ids)
    precision = 1.0 / 10 if true_song_id in pred_ids else 0.0
    hit = int(true_song_id in pred_ids)
    recall_at_10.append(recall)
    precision_at_10.append(precision)
    hit_at_10.append(hit)
recall10 = np.mean(recall_at_10)
precision10 = np.mean(precision_at_10)
hitrate10 = np.mean(hit_at_10)
print(f"Recall@10: {recall10:.4f}")
print(f"Precision@10: {precision10:.4f}")
print(f"Hit Rate@10: {hitrate10:.4f}")


--- Evaluasi Model pada Data Validasi ---
Recall@10: 0.1710
Precision@10: 0.0171
Hit Rate@10: 0.1710
Recall@10: 0.1710
Precision@10: 0.0171
Hit Rate@10: 0.1710


# Tuning Arsitektur, Batch Size, dan Augmentasi Data

**Perbaikan tambahan:**

- Arsitektur model diperbesar (hidden_dim, n_layers, output_embedding_dim).
- Batch size diperbesar untuk stabilitas training.
- Augmentasi data sederhana: random deletion dan random swap pada kata di pesan.


In [20]:
# Update konfigurasi model dan batch size
tuned_config = CONFIG.copy()
tuned_config["hidden_dim"] = 512
# Tambah layer GRU
tuned_config["n_layers"] = 3
tuned_config["output_embedding_dim"] = 128
tuned_config["batch_size"] = 256

print("Konfigurasi baru:", tuned_config)

Konfigurasi baru: {'text_embedding_dim': 150, 'hidden_dim': 512, 'n_layers': 3, 'dropout': 0.5, 'output_embedding_dim': 128, 'batch_size': 256, 'learning_rate': 0.001, 'epochs': 25, 'max_len': 50, 'margin': 0.2, 'min_song_freq': 5}


In [21]:
# Augmentasi data: random deletion dan random swap

def random_deletion(words, p=0.1):
    if len(words) == 1:
        return words
    return [w for w in words if random.random() > p or len(words) == 1]

def random_swap(words, n=1):
    words = words.copy()
    for _ in range(n):
        idx1, idx2 = random.sample(range(len(words)), 2)
        words[idx1], words[idx2] = words[idx2], words[idx1]
    return words

def augment_text(text):
    words = text.split()
    # Random deletion
    words = random_deletion(words, p=0.1)
    # Random swap
    if len(words) > 1:
        words = random_swap(words, n=1)
    return ' '.join(words)

# Terapkan augmentasi pada data train
train_df["augmented_message"] = train_df["cleaned_message"].apply(augment_text)
X_train_aug = list(train_df["cleaned_message"]) + list(train_df["augmented_message"])
y_train_aug = list(train_df["song_name"]) + list(train_df["song_name"])
print(f"Data train setelah augmentasi: {len(X_train_aug)} sample")

Data train setelah augmentasi: 72064 sample


In [22]:
# Buat ulang dataset dan dataloader dengan konfigurasi baru dan data augmentasi
train_dataset = RecommendationDataset(X_train_aug, y_train_aug, vocab, song_map, tuned_config["max_len"])
train_loader = DataLoader(train_dataset, batch_size=tuned_config["batch_size"], shuffle=True)

# Inisialisasi ulang model dengan arsitektur baru
torch.cuda.empty_cache()
message_tower = MessageTower(vocab_size, tuned_config)
song_tower = SongTower(num_songs, tuned_config)

# Training ulang dengan konfigurasi baru
def train_recommendation_model_tuned(message_tower, song_tower, train_loader, config, device):
    message_tower.to(device)
    song_tower.to(device)
    optimizer = optim.AdamW(
        list(message_tower.parameters()) + list(song_tower.parameters()),
        lr=config["learning_rate"]
    )
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)
    criterion = nn.CosineEmbeddingLoss(margin=config["margin"])
    all_song_ids = torch.arange(len(song_map)).to(device)
    for epoch in range(1, config["epochs"] + 1):
        start_time = time.time()
        message_tower.train()
        song_tower.train()
        total_loss = 0
        for messages, positive_songs in train_loader:
            messages, positive_songs = messages.to(device), positive_songs.to(device)
            optimizer.zero_grad()
            message_embeddings = message_tower(messages)
            song_embeddings = song_tower(positive_songs)
            # In-batch negative
            loss_pos = criterion(message_embeddings, song_embeddings, torch.ones(messages.size(0)).to(device))
            negative_song_embeddings = torch.roll(song_embeddings, shifts=1, dims=0)
            loss_neg = criterion(message_embeddings, negative_song_embeddings, -torch.ones(messages.size(0)).to(device))
            # Random hard negative
            rand_neg_ids = torch.randint(0, len(song_map), (messages.size(0),)).to(device)
            rand_neg_emb = song_tower(rand_neg_ids)
            loss_rand_neg = criterion(message_embeddings, rand_neg_emb, -torch.ones(messages.size(0)).to(device))
            loss = loss_pos + loss_neg + 0.5 * loss_rand_neg
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        scheduler.step()
        avg_loss = total_loss / len(train_loader)
        end_time = time.time()
        print(f"Epoch: {epoch}/{config['epochs']} | Waktu: {end_time - start_time:.2f}s | Average Loss: {avg_loss:.4f} | LR: {scheduler.get_last_lr()[0]:.6f}")

train_recommendation_model_tuned(message_tower, song_tower, train_loader, tuned_config, device)

Epoch: 1/25 | Waktu: 85.12s | Average Loss: 0.8742 | LR: 0.001000
Epoch: 2/25 | Waktu: 75.95s | Average Loss: 0.8174 | LR: 0.001000
Epoch: 3/25 | Waktu: 83.81s | Average Loss: 0.7934 | LR: 0.001000
Epoch: 4/25 | Waktu: 90.15s | Average Loss: 0.7733 | LR: 0.001000
Epoch: 5/25 | Waktu: 82.95s | Average Loss: 0.7579 | LR: 0.000500
Epoch: 6/25 | Waktu: 84.78s | Average Loss: 0.7458 | LR: 0.000500
Epoch: 7/25 | Waktu: 83.39s | Average Loss: 0.7394 | LR: 0.000500
Epoch: 8/25 | Waktu: 82.37s | Average Loss: 0.7305 | LR: 0.000500
Epoch: 9/25 | Waktu: 83.49s | Average Loss: 0.7251 | LR: 0.000500
Epoch: 10/25 | Waktu: 85.34s | Average Loss: 0.7184 | LR: 0.000250
Epoch: 11/25 | Waktu: 84.78s | Average Loss: 0.7083 | LR: 0.000250
Epoch: 12/25 | Waktu: 85.31s | Average Loss: 0.7036 | LR: 0.000250
Epoch: 13/25 | Waktu: 80.89s | Average Loss: 0.6970 | LR: 0.000250
Epoch: 14/25 | Waktu: 81.23s | Average Loss: 0.6938 | LR: 0.000250
Epoch: 15/25 | Waktu: 81.53s | Average Loss: 0.6892 | LR: 0.000125
Epoc

# Average Loss pada Validation Set (Model Tuning)

Cell ini menghitung average loss (CosineEmbeddingLoss) pada data validasi untuk model hasil tuning.


In [23]:
# Hitung average loss pada validation set untuk model tuning
def compute_validation_loss_tuned(message_tower, song_tower, val_dataset, config, device):
    message_tower.eval()
    song_tower.eval()
    criterion = nn.CosineEmbeddingLoss(margin=config["margin"])
    val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False)
    total_loss = 0
    n = 0
    with torch.no_grad():
        for messages, positive_songs in val_loader:
            messages, positive_songs = messages.to(device), positive_songs.to(device)
            message_embeddings = message_tower(messages)
            song_embeddings = song_tower(positive_songs)
            # Loss untuk pasangan positif
            loss_pos = criterion(message_embeddings, song_embeddings, torch.ones(messages.size(0)).to(device))
            # Loss untuk pasangan negatif (shift)
            negative_song_embeddings = torch.roll(song_embeddings, shifts=1, dims=0)
            loss_neg = criterion(message_embeddings, negative_song_embeddings, -torch.ones(messages.size(0)).to(device))
            # Random hard negative
            rand_neg_ids = torch.randint(0, len(song_map), (messages.size(0),)).to(device)
            rand_neg_emb = song_tower(rand_neg_ids)
            loss_rand_neg = criterion(message_embeddings, rand_neg_emb, -torch.ones(messages.size(0)).to(device))
            loss = loss_pos + loss_neg + 0.5 * loss_rand_neg
            total_loss += loss.item() * messages.size(0)
            n += messages.size(0)
    avg_loss = total_loss / n
    print(f"Average validation loss (tuned model): {avg_loss:.4f}")
    return avg_loss

compute_validation_loss_tuned(message_tower, song_tower, val_dataset, tuned_config, device)

Average validation loss (tuned model): 0.7522


0.7522249612814579